In [ ]:
# Customer Churn Prediction using Machine Learning
import pandas as pd
import numpy as np


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


In [15]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Load Dataset
df = pd.read_csv("Telco-Customer-Churn_IBM.csv")


In [16]:

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())


Dataset shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingM

In [17]:

# 2. Data Cleaning
df.drop_duplicates(inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df.drop(columns=["customerID"], inplace=True)


In [18]:

# 3. Prepare Features and Target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df.dropna(subset=["Churn"], inplace=True)

X = df.drop(columns=["Churn"])
y = df["Churn"].astype(int)


In [19]:

# 4. Identify Column Types
numeric_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "bool", "category"]
).columns.tolist()

# 5. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

# 6. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=72,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

# 7. Build Machine Learning Pipeline
model = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(
        RandomForestClassifier(
            n_estimators=100,
            random_state=37
        ),
        threshold="median"
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# 8. Train Model
model.fit(X_train, y_train)

# 9. Make Predictions
y_pred = model.predict(X_test)

# 10. Evaluate Model
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Training data: (5634, 19)
Testing data: (1409, 19)
Model Accuracy: 0.809

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.89      0.87      1035
           1       0.65      0.59      0.62       374

    accuracy                           0.81      1409
   macro avg       0.76      0.74      0.75      1409
weighted avg       0.80      0.81      0.81      1409

Confusion Matrix:
[[918 117]
 [152 222]]
